In [ ]:
import numpy as np
import pandas as pd
from functools import partial, reduce
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import warnings

In [ ]:
warnings.filterwarnings("ignore")
df = pd.read_parquet("data/processed/base_EPEX_df.parquet")
df.head()

In [ ]:
# implements linear regression for each hour as its own model

target = "Price (GBP/MWhe)"

# reshape from hourly rows to one row per day 
d = df.sort_values("Hour").copy()
d["date"] = pd.to_datetime(d["Hour"].dt.date)   # naive local calendar date
d["hod"] = d["Hour"].dt.hour

# Keep only complete 24-hour days (drops DST days with 23/25 hours and partial days)
d = d[d.groupby("date")["Hour"].transform("size") == 24]

def wide(col, prefix):
    w = d.pivot(index="date", columns="hod", values=col)
    w.columns = [f"{prefix}_h{h:02d}" for h in w.columns]
    return w

demand = wide("FORECASTDEMAND", "demand")
wind = wide("Incentive_forecast", "wind")
oil = wide("DCOILBRENTEU", "oil")
gas = wide("sap_lag2d", "gas")
price = wide(target, "price")

# Autoregressive features: shift(freq="D") shifts by true calendar days,
# so missing days produce NaN rather than silently misaligned rows
price_lag1 = price.shift(1, freq="D").add_suffix("_lag1d")   # yesterday's 24 prices

X = (demand.join(wind, how="inner")
           .join(gas, how="inner")
           .join(oil, how="inner")
           .join(price_lag1, how="inner"))

X.index = pd.to_datetime(X.index)
X["day_of_week"] = X.index.dayofweek
X["month"] = X.index.month_name()
X = X.dropna()
Y = price.loc[X.index]    # 24 columns: the target vector for each day

In [ ]:
X.to_parquet("data/processed/X.parquet")
Y.to_parquet("data/processed/Y.parquet")

In [53]:
# pipeline 

split = int(len(X) * 0.85)
X_train, Y_train = X.iloc[:split], Y.iloc[:split]
X_test, Y_test = X.iloc[split:], Y.iloc[split:]

numeric_features = [c for c in X_train.columns if c not in ["month", "day_of_week"]]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["day_of_week", "month"]),
])

baseline = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),   # multi-output: fits one linear model per hour
])

ridge = Pipeline([
    ("prep", preprocess),
    ("reg", Ridge()),   # multi-output: fits one linear model per hour
])

lgbm = Pipeline([
    ("prep", preprocess),
    ("reg", MultiOutputRegressor(
        LGBMRegressor(n_estimators=100, learning_rate=0.05, verbosity=-1)
    )),
])

# --- Time-ordered cross-validation ---
tscv = TimeSeriesSplit(n_splits=5)
rmses_base = []
rmses_ridge = []
rmses_lgbm = []

for train_idx, test_idx in tscv.split(X_train):
    baseline.fit(X_train.iloc[train_idx], Y_train.iloc[train_idx])
    ridge.fit(X_train.iloc[train_idx], Y_train.iloc[train_idx])
    lgbm.fit(X_train.iloc[train_idx], Y_train.iloc[train_idx])

    pred_base = baseline.predict(X_train.iloc[test_idx])
    pred_ridge = ridge.predict(X_train.iloc[test_idx])
    pred_lgbm = lgbm.predict(X_train.iloc[test_idx])

    rmses_base.append(np.sqrt(np.mean((Y_train.iloc[test_idx].values - pred_base) ** 2)))
    rmses_ridge.append(np.sqrt(np.mean((Y_train.iloc[test_idx].values - pred_ridge) ** 2)))
    rmses_lgbm.append(np.sqrt(np.mean((Y_train.iloc[test_idx].values - pred_lgbm) ** 2)))

print("CV RMSE per fold (baseline):", np.round(rmses_base, 2))
print("CV RMSE per fold (ridge):", np.round(rmses_ridge, 2))
print("CV RMSE per fold (lgbm):", np.round(rmses_lgbm, 2))

CV RMSE per fold (baseline): [ 8.92 58.87 71.58 17.8  24.77]
CV RMSE per fold (ridge): [ 7.87 58.72 70.39 16.34 23.47]
CV RMSE per fold (lgbm): [10.93 82.52 90.69 19.09 21.08]


In [54]:
# checking on test data

baseline.fit(X_train, Y_train)
ridge.fit(X_train, Y_train)
lgbm.fit(X_train, Y_train)

pred_base = pd.DataFrame(baseline.predict(X_test), index=Y_test.index, columns=Y_test.columns)
pred_ridge = pd.DataFrame(ridge.predict(X_test), index=Y_test.index, columns=Y_test.columns)
pred_lgbm = pd.DataFrame(lgbm.predict(X_test), index=Y_test.index, columns=Y_test.columns)

err_base = pred_base - Y_test
err_ridge = pred_ridge - Y_test
err_lgbm = pred_lgbm - Y_test
print("Baseline MAE :", err_base.abs().values.mean())
print("Ridge MAE :", err_ridge.abs().values.mean())
print("LGBM MAE:", err_lgbm.abs().values.mean())

Baseline MAE : 12.354351768542685
Ridge MAE : 11.962033808929004
LGBM MAE: 12.03965291521111


In [55]:
print("Baseline RMSE:", np.sqrt((err_base ** 2).values.mean()))
print("Ridge RMSE:", np.sqrt((err_ridge ** 2).values.mean()))
print("LGBM RMSE:", np.sqrt((err_lgbm ** 2).values.mean()))

Baseline RMSE: 18.223381056026888
Ridge RMSE: 17.721218993677738
LGBM RMSE: 19.02505316755618


In [56]:
def metrics(y_true, y_pred):
    err = np.asarray(y_pred) - np.asarray(y_true)
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean())}

naive_prev_day = X_test[[f"{c}_lag1d" for c in Y.columns]].values

results = pd.DataFrame({
    "Regression": metrics(Y_test, pred_base),
    "Ridge": metrics(Y_test, pred_ridge),
    "Prev day":   metrics(Y_test, naive_prev_day),
    "LGBM": metrics(Y_test, pred_lgbm)
}).T
print(results.round(2))

# Per-hour MAE
by_hour = pd.DataFrame({
    "Regression": (pred_base - Y_test).abs().mean().values,
    "Ridge": (pred_ridge - Y_test).abs().mean().values,
    "LGBM" : (pred_lgbm - Y_test).abs().mean().values,
    "Prev day":   np.abs(naive_prev_day - Y_test.values).mean(axis=0),
}, index=range(24))
print(by_hour.round(2))

# Improvement over baseline (positive = regression is better)
over_easy = 1 - results.loc["Ridge", "RMSE"] / results.loc["Prev day", "RMSE"]
over_base = 1 - results.loc["Ridge", "RMSE"] / results.loc["Regression", "RMSE"]
over_ridge = 1 - results.loc["Ridge", "RMSE"] / results.loc["LGBM", "RMSE"]
print(f"RMSE improvement of ridge vs prev-day baseline: {over_easy:.1%}")
print(f"RMSE improvement of ridge vs regression baseline: {over_base:.1%}")
print(f"RMSE improvement of ridge vs LGBM baseline: {over_ridge:.1%}")

              MAE   RMSE
Regression  12.35  18.22
Ridge       11.96  17.72
Prev day    18.25  30.10
LGBM        12.04  19.03
    Regression  Ridge   LGBM  Prev day
0         3.95   3.64   3.84     15.40
1         4.90   4.50   4.62     15.27
2         6.51   6.06   6.86     15.26
3         7.62   7.24   8.31     15.27
4         8.38   7.88   9.16     15.67
5         9.89   9.29  11.40     17.28
6         9.95   9.29  10.32     19.39
7        11.92  11.71  11.79     21.35
8        14.25  14.06  13.36     22.08
9        14.58  14.34  14.94     22.69
10       14.34  14.24  16.35     22.86
11       14.70  14.57  17.23     22.88
12       14.96  14.87  14.70     23.28
13       15.13  14.98  14.46     23.02
14       15.06  14.95  15.02     22.48
15       14.02  13.94  15.10     21.14
16       15.00  14.44  13.69     18.86
17       21.60  19.94  14.85     17.13
18       17.69  17.19  15.71     17.43
19       16.21  15.40  13.48     15.84
20       11.93  11.68  11.23     13.35
21       10.24  1